In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
marvel = pd.read_csv("marvel-wikia-data.csv")
dc = pd.read_csv("dc-wikia-data.csv") 

marvel["publisher"] = "Marvel"
dc["publisher"] = "DC"

df = pd.concat([marvel, dc], ignore_index=True)

initial_count = len(df)
initial_count
# I import the necessary libraries to run the project. 
# I join the two data sources, Marvel and DC csv files so that they can run at the same time.
# I then run the information to get a handle on what I am looking at.

In [ ]:
df = df.drop(columns="GSM")
after_drop_gsm = len(df)
after_drop_gsm


# Dropped unnecessary column
# Checked DF afterwards to see what happened and verify dropping

In [ ]:
# 1. Drop rows with missing names
df = df.dropna(subset=["name"])

# 2. Convert YEAR to numeric and drop invalid or missing years
df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")
df = df[df["YEAR"].notna()]

# 3. Drop characters with no appearances
df = df[df["APPEARANCES"] > 0]

# 4. Keep only Male and Female characters
df = df[df["SEX"].isin(["Male Characters", "Female Characters"])]

# 5. Remove duplicate characters (same name + publisher)
df = df.drop_duplicates(subset=["name", "publisher"])

# 6. Strip whitespace from names
df["name"] = df["name"].str.strip()

df.head()


In [ ]:

# This line creates an array of 100 random integers between 1 and 10.
# Each number represents a "character" category for the histogram.
# Think of it like: each character gets assigned a number from 1 to 10.

x = np.random.randint(1, 10, size=100)


# This line creates 100 random decimal numbers between 0 and 1.
# These numbers will be used as "weights" in the histogram.
# A weight controls how tall each bar is for each data point.
# In simple terms: higher weight = taller bar.

y = np.random.rand(100)


# - x is the data being grouped into bars.
# - bins=8 means the data will be split into 8 groups.
# - weights=y means each x-value contributes its weight to the bar height.
# - color='black' makes the bars black.
# - edgecolor='red' outlines each bar in red.

plt.hist(x, bins=8, weights=y, color='black', edgecolor='red')


# This labels the x-axis of the plot.
# It tells the viewer what the horizontal axis represents.

plt.xlabel("Characters")

# ------------------------------------------------------------
# This labels the y-axis.
# It explains what the bar heights represent.
# ------------------------------------------------------------
plt.ylabel("Appearances by Character")

# ------------------------------------------------------------
# This sets the title at the top of the plot.
# It describes what the entire chart is showing.
# ------------------------------------------------------------
plt.title("Frequency of Appearance by Male and Female Villains and Heroes")

# ------------------------------------------------------------
# This displays the final histogram on the screen.
# Without this line, nothing would appear.
# ------------------------------------------------------------
plt.show()


In [ ]:
# Create grouping label from existing cleaned columns
filtered = df.copy()

filtered["Group"] = (
    filtered["SEX"].str.extract(r"(Male|Female)") + " " +
    filtered["ALIGN"].str.extract(r"(Good|Bad)")
)

# Drop rows where the regex didn't match (should be rare if cleaned)
filtered = filtered.dropna(subset=["Group"])

# Plot
plt.figure(figsize=(12, 7))
sns.boxplot(
    data=filtered,
    x="Group",
    y="APPEARANCES",
    hue="Group",
    palette="Blues",
    legend=False,
    showfliers=False
)

plt.yscale("log")
plt.title("Distribution of Comic Appearances by Gender and Alignment", fontsize=18, weight="bold")
plt.xlabel("Character Type", fontsize=14)
plt.ylabel("Appearances (Log Scale)", fontsize=14)
plt.xticks(rotation=20, fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Copy the cleaned DataFrame
plot_df = df.copy()

# Map Good → Hero, Bad → Villain (no other categories included)
alignment_map = {
    "Good Characters": "Heroes",
    "Bad Characters": "Villains"
}

plot_df["ALIGN"] = plot_df["ALIGN"].map(alignment_map)

# Drop anything that didn't map (Neutral, Reformed, Unknown, etc.)
plot_df = plot_df.dropna(subset=["ALIGN"])

# Build pivot table for heatmap
heatmap_data = (
    plot_df.pivot_table(
        index="SEX",
        columns="ALIGN",
        values="APPEARANCES",
        aggfunc="mean"
    )
)

# Plot heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    linewidths=0.5
)

plt.title("Average Comic Appearances: Heroes vs Villains by Gender", fontsize=18, weight="bold")
plt.xlabel("Alignment", fontsize=14)
plt.ylabel("Gender", fontsize=14)
plt.xticks(rotation=20)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()
